# **Day 38: Raising Exceptions and Custom Exceptions**

Yesterday we **caught** errors. Today we learn to **raise** (create) our own errors when something is wrong.

We use the `raise` keyword. It is like a referee showing a **red card**: the program cannot continue normally.

In [1]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError("Age must be a whole number")
    if age < 0 or age > 130:
        raise ValueError(f"Invalid age: {age}")
    return age

print(set_age(25))

try:
    set_age(-3)
except ValueError as e:
    print("Caught:", e)

25
Caught: Invalid age: -3


## **Example from GIS: Checking Latitude and Longitude**
- **Latitude** must be between **-90 and +90**.
- **Longitude** must be between **-180 and +180**.

A very common mistake is to **swap** latitude and longitude. For West Bengal, latitude is around 22 to 27 and longitude is around 86 to 90. If someone writes 88 as latitude, it is clearly wrong.

In [2]:
def validate_point(lat, lon):
    if not -90 <= lat <= 90:
        raise ValueError(f"Latitude {lat} is outside -90 to 90 (did you swap lat and lon?)")
    if not -180 <= lon <= 180:
        raise ValueError(f"Longitude {lon} is outside -180 to 180")
    return lat, lon

for lat, lon in [(22.57, 88.36), (88.36, 22.57), (26.7, 200)]:
    try:
        validate_point(lat, lon)
        print(f"({lat}, {lon}) is valid")
    except ValueError as e:
        print("Error:", e)

(22.57, 88.36) is valid
(88.36, 22.57) is valid
Error: Longitude 200 is outside -180 to 180


## **Custom Exceptions**
We can make our **own error types** by making a class that comes from `Exception`. (We will learn classes properly on Day 41. For now, just see the pattern.)

Custom errors make the code clearer: the error name itself tells what went wrong.

In [3]:
class InsufficientBalance(Exception):
    pass

class InvalidPin(Exception):
    pass

balance = 1500

def withdraw(amount, pin):
    global balance
    if pin != "2580":
        raise InvalidPin("Wrong PIN")
    if amount > balance:
        raise InsufficientBalance(f"Balance is {balance}, but asked for {amount}")
    balance -= amount
    return balance

for amount, pin in [(500, "2580"), (5000, "2580"), (100, "1111")]:
    try:
        left = withdraw(amount, pin)
        print(f"Withdrew {amount}, balance left {left}")
    except InsufficientBalance as e:
        print("Insufficient balance:", e)
    except InvalidPin as e:
        print("PIN error:", e)

Withdrew 500, balance left 1000
Insufficient balance: Balance is 1000, but asked for 5000
PIN error: Wrong PIN


## **Example from Remote Sensing: A Cloud Cover Error**

In [4]:
class TooCloudyError(Exception):
    pass

def load_scene(scene_id, cloud_cover, max_cloud=20):
    if cloud_cover > max_cloud:
        raise TooCloudyError(f"{scene_id} has {cloud_cover}% cloud (limit is {max_cloud}%)")
    return f"{scene_id} loaded"

scenes = [("S2A_20240705", 85), ("S2B_20241110", 4), ("S2A_20240820", 60)]
for sid, cloud in scenes:
    try:
        print(load_scene(sid, cloud))
    except TooCloudyError as e:
        print("Skipped:", e)

Skipped: S2A_20240705 has 85% cloud (limit is 20%)
S2B_20241110 loaded
Skipped: S2A_20240820 has 60% cloud (limit is 20%)


## **Raising the Error Again**
Sometimes we want to **note** an error and still let it go up to the caller. We write just `raise`.

In [5]:
def parse_pin(s):
    try:
        return int(s)
    except ValueError:
        print("Log: bad PIN code", repr(s))
        raise

try:
    parse_pin("7000O1")        # Letter O instead of zero
except ValueError as e:
    print("The caller also got the error:", e)

Log: bad PIN code '7000O1'
The caller also got the error: invalid literal for int() with base 10: '7000O1'


## **assert: A Quick Check**
`assert condition, "message"` raises an `AssertionError` if the condition is False. It is used to check our own assumptions while writing code.

In [6]:
def percentage(obtained, total):
    assert total > 0, "Total must be positive"
    return obtained / total * 100

print(percentage(45, 50))
# percentage(45, 0)         # This will raise an AssertionError

90.0


## **Practice Problem: Validate a West Bengal Vehicle Number**
Format: `WB 02 AB 1234` (WB, 2 digits, 1 or 2 letters, 4 digits). We will do this more easily with regex on Day 63.

In [7]:
class InvalidVehicleNumber(ValueError):
    pass

def validate_vehicle(num):
    parts = num.upper().split()
    if len(parts) != 4:
        raise InvalidVehicleNumber(f"{num!r}: need 4 parts")
    state, district, series, number = parts
    if state != "WB":
        raise InvalidVehicleNumber(f"{num!r}: not a West Bengal number")
    if not (district.isdigit() and len(district) == 2):
        raise InvalidVehicleNumber(f"{num!r}: district code must be 2 digits")
    if not (series.isalpha() and 1 <= len(series) <= 2):
        raise InvalidVehicleNumber(f"{num!r}: series must be 1 or 2 letters")
    if not (number.isdigit() and len(number) == 4):
        raise InvalidVehicleNumber(f"{num!r}: last part must be 4 digits")
    return True

for v in ["WB 02 AB 1234", "wb 74 c 0042", "OD 02 AB 1234", "WB 2 AB 1234", "WB 06 AB 12"]:
    try:
        validate_vehicle(v)
        print("Valid  :", v)
    except InvalidVehicleNumber as e:
        print("Invalid:", e)

Valid  : WB 02 AB 1234
Valid  : wb 74 c 0042
Invalid: 'OD 02 AB 1234': not a West Bengal number
Invalid: 'WB 2 AB 1234': district code must be 2 digits
Invalid: 'WB 06 AB 12': last part must be 4 digits


In [8]:
# Quick question
# Which message will be printed?
# try:
#     raise ValueError("bad value")
# except TypeError:
#     print("type problem")
# except ValueError as e:
#     print("value problem:", e)